In [ ]:
import pyodbc
import pandas as pd
import numpy as np
# CONFIGURATION / PROCESSING PARAMETERS
# Allowed range for the final mandrel length.
MIN_MANDREL_LENGTH=40.3
MAX_MANDREL_LENGTH=40.8
# Nominal mandrel length used when calculating theoretical mandrel boundaries.
MANDREL_LENGTH=40.8
# If the final mandrel is shorter than this value, it is removed.
MIN_LAST_MANDREL_LENGTH=25.0
# Only production data from this date onward is processed.
START_DATE="2025-01-01 00:00:00"
# Time window around a theoretical mandrel boundary in which an OD drop can be searched for.
BOUNDARY_SEARCH_MINUTES=2
# Minimum OD decrease considered to be a significant drop.
MIN_OD_DROP=0.05
# Multiplier used with MAD (Median Absolute Deviation) to dynamically determine the OD-drop threshold.
DROP_MAD_MULTIPLIER=6.0
# OD value above this threshold is considered to indicate active production.
STOP_OD_THRESHOLD=1.0
# Minimum duration of an interruption required before the initial short mandrel is removed.
MIN_STOP_DURATION_SECONDS=60
# Maximum allowed time between the beginning of production and the first interruption.
INITIAL_INTERRUPTION_MAX_SECONDS=200
# Initial short-mandrel removal is only attempted when at least this many mandrels have been detected.
MIN_MANDREL_COUNT_FOR_REMOVAL=6
# Number of rows read from SQL Server at a time.
CHUNK_SIZE=50000
# Number of rows inserted in each database batch.
INSERT_BATCH_SIZE=100
# SQL SERVER CONNECTION
CONNECTION_STRING=("DRIVER={ODBC Driver 18 for SQL Server};SERVER=atwpSQL-STP-app;DATABASE=LinePC7442;Trusted_Connection=yes;TrustServerCertificate=yes;")
"""
# SOURCE DATA QUERY
# Loads production data for one Prog_Nr + ordername combination.
#
# Important calculated fields:
#   production_m   -> raw production length calculated from line speed
#   OD_real        -> actual outer diameter
#   ID_real        -> actual inner diameter
#   wt_real        -> average wall thickness
#
# The recipe description is also split into:
#   compound
#   ID
#   wth
"""
SOURCE_QUERY="""
SELECT
t1.[timestamp],
LEFT(CAST(t1.[Prog_Nr] AS VARCHAR(MAX)),255) AS Prog_Nr,
t1.[ordername],
t1.[Linie_ist],
(t1.[Linie_ist]/60.0)*2 AS production_m,
t1.[Auszen_DM_XY_ist] AS OD_real,
t1.[Extr_ist] AS rpm,
t1.[Innen_DM_ist] AS ID_real,
t1.[Massedruck] AS mass_pressure,
t1.[Mass_ist] AS mass_temp,
(ISNULL(t1.[Wand_X_links],0)+ISNULL(t1.[Wand_X_rechts],0)+ISNULL(t1.[Wand_Y_links],0)+ISNULL(t1.[Wand_Y_rechts],0))/4.0 AS wt_real,
CASE
WHEN LEN(tr.[description])-LEN(REPLACE(tr.[description],'_',''))>=2
THEN LEFT(tr.[description],CHARINDEX('_',tr.[description])-1)
ELSE tr.[description]
END AS compound,
CASE
WHEN LEN(tr.[description])-LEN(REPLACE(tr.[description],'_',''))>=2
THEN SUBSTRING(tr.[description],CHARINDEX('_',tr.[description])+1,CHARINDEX('_',tr.[description],CHARINDEX('_',tr.[description])+1)-CHARINDEX('_',tr.[description])-1)
ELSE NULL
END AS ID,
CASE
WHEN LEN(tr.[description])-LEN(REPLACE(tr.[description],'_',''))>=2
THEN SUBSTRING(tr.[description],CHARINDEX('_',tr.[description],CHARINDEX('_',tr.[description])+1)+1,LEN(tr.[description]))
ELSE NULL
END AS wth
FROM dbo.DI_Ex2_Istwerte t1
LEFT JOIN dbo.troester_recipe tr
ON CAST(t1.[Prog_Nr] AS VARCHAR(255))=CAST(tr.[recipename] AS VARCHAR(255))
WHERE t1.[timestamp]>=?
AND CAST(t1.[Prog_Nr] AS VARCHAR(255))=?
AND t1.[ordername]=?
ORDER BY t1.[timestamp]
"""
# GET ALL AVAILABLE PROGRAM / ORDER COMBINATIONS
ORDERS_QUERY="""
SELECT DISTINCT
CAST([Prog_Nr] AS VARCHAR(255)) AS Prog_Nr,
[ordername]
FROM dbo.DI_Ex2_Istwerte
WHERE [timestamp]>=?
AND [Prog_Nr] IS NOT NULL
AND [ordername] IS NOT NULL
AND LTRIM(RTRIM(CAST([Prog_Nr] AS VARCHAR(255))))<>''
AND LTRIM(RTRIM(CAST([ordername] AS VARCHAR(255))))<>''
ORDER BY CAST([Prog_Nr] AS VARCHAR(255)),[ordername]
"""
# EXISTING TARGET ROWS
# Used to determine which production rows have already been inserted into the report table.
EXISTING_ROWS_QUERY="""
SELECT
[timestamp],
CAST([Prog_Nr] AS VARCHAR(255)) AS Prog_Nr,
[ordername]
FROM dbo.Ex2_report_dev
WHERE [timestamp]>=?
AND [Prog_Nr] IS NOT NULL
AND [ordername] IS NOT NULL
"""
# MAIN PROCESSOR CLASS
class MandrelProcessor:
    def __init__(self,connection):
        # Reuse one database connection for the complete run.
        self.connection=connection
        # Continue Pos numbering from the current target table.
        self.next_pos=self.get_next_pos()
        # Set of already-existing rows.
        self.existing_keys=None
    """
    # CLEAR TARGET TABLE
    # WARNING:
    # This deletes ALL rows from the report table.
    # It is currently disabled in main().
    """
    def clear_target_table(self):
        cursor=self.connection.cursor()
        cursor.execute("TRUNCATE TABLE dbo.Ex2_report_dev")
        self.connection.commit()
        cursor.close()
        # Restart Pos numbering after truncation.
        self.next_pos=1
    # GET NEXT POS NUMBER
    def get_next_pos(self):
        cursor=self.connection.cursor()
        # Continue Pos numbering from the highest existing Pos.
        cursor.execute("SELECT ISNULL(MAX([Pos]),0)+1 FROM dbo.Ex2_report_dev")
        value=cursor.fetchone()[0]
        cursor.close()
        return int(value)
    """
    # LOAD EXISTING ROW KEYS
    # A row is uniquely identified here by:
    #   timestamp + Prog_Nr + ordername
    """
    def get_existing_keys(self):
        df=pd.read_sql(EXISTING_ROWS_QUERY,self.connection,params=[START_DATE])
        if df.empty:
            return set()
        # Normalize data types before creating the key set.
        df["timestamp"]=pd.to_datetime(df["timestamp"],errors="coerce")
        df["Prog_Nr"]=df["Prog_Nr"].fillna("").astype(str).str.strip()
        df["ordername"]=df["ordername"].fillna("").astype(str).str.strip()
        df=df.dropna(subset=["timestamp"])
        # Convert the DataFrame into a fast lookup set.
        return set(zip(df["timestamp"],df["Prog_Nr"],df["ordername"]))
    # GET PROGRAM / ORDER PAIRS
    def get_prog_order_pairs(self):
        df=pd.read_sql(ORDERS_QUERY,self.connection,params=[START_DATE])
        # Normalize string fields.
        df["Prog_Nr"]=df["Prog_Nr"].fillna("").astype(str).str.strip()
        df["ordername"]=df["ordername"].fillna("").astype(str).str.strip()
        # Remove invalid program/order combinations.
        df=df[(df["Prog_Nr"]!="")&(df["ordername"]!="")]
        # Ensure each program/order combination is processed once.
        df=df.drop_duplicates(["Prog_Nr","ordername"])
        # Process combinations in a predictable order.
        df=df.sort_values(["Prog_Nr","ordername"]).reset_index(drop=True)
        return df
    # LOAD ONE ORDER'S RAW PRODUCTION DATA
    def load_order_data(self,program,order):
        # Read large orders in chunks to reduce memory usage.
        chunks=pd.read_sql(SOURCE_QUERY,self.connection,params=[START_DATE,program,order],chunksize=CHUNK_SIZE)
        # Combine all chunks into one DataFrame.
        df=pd.concat(chunks,ignore_index=True)
        if df.empty:
            return df
        # Convert timestamp to pandas datetime.
        df["timestamp"]=pd.to_datetime(df["timestamp"],errors="coerce")
        # Convert recipe ID from comma decimal notation to numeric.
        df["ID"]=df["ID"].astype(str).str.replace(",",".",regex=False)
        df["ID"]=pd.to_numeric(df["ID"],errors="coerce")
        df["wth"]=df["wth"].astype(str).str.replace(",",".",regex=False)
        df["wth"]=pd.to_numeric(df["wth"],errors="coerce")
        # production_m is the raw production length for this 2-second measurement.
        df["production_m"]=pd.to_numeric(df["production_m"],errors="coerce")
        # Replace infinity values with NaN.
        df["production_m"]=df["production_m"].replace([np.inf,-np.inf],np.nan)
        # Negative production lengths are invalid.
        df.loc[df["production_m"]<0,"production_m"]=np.nan
        # Make sure chronological processing is guaranteed.
        df=df.sort_values("timestamp").reset_index(drop=True)
        return df
    # CALCULATE LENGTH BETWEEN TWO TIMESTAMPS
    def get_segment_length(self,df,start_time,end_time):
        # Select only rows belonging to the requested time interval.
        segment=df[(df["timestamp"]>=start_time)&(df["timestamp"]<end_time)]
        # Sum raw production length.
        return segment["production_m"].fillna(0).sum()
    # CALCULATE FINAL LENGTH OF EACH MANDREL
    def calculate_final_mandrel_lengths(self,df):
        # Keep the columns required for the calculation.
        result=df[["mandrel_no","production_m"]].copy()
        result["mandrel_no"]=pd.to_numeric(result["mandrel_no"],errors="coerce")
        result["production_m"]=pd.to_numeric(result["production_m"],errors="coerce")
        result=result.dropna(subset=["mandrel_no","production_m"])
        if result.empty:
            return pd.DataFrame(columns=["mandrel_no","final_mandrel_m"])
        result["mandrel_no"]=result["mandrel_no"].astype(int)
        # Sum all raw production length belonging to each mandrel.
        final_lengths=result.groupby("mandrel_no",as_index=False)["production_m"].sum().rename(columns={"production_m":"final_mandrel_m"})
        # Store final lengths with two decimal places.
        final_lengths["final_mandrel_m"]=final_lengths["final_mandrel_m"].round(2)
        final_lengths=final_lengths.sort_values("mandrel_no").reset_index(drop=True)
        return final_lengths
    """
    # INITIAL MANDREL NUMBER CALCULATION
    # The algorithm works backwards from the end of production.
    # Every MANDREL_LENGTH metres corresponds to one mandrel.
    """
    def calculate_initial_mandrels(self,df):
        df=df.sort_values("timestamp").reset_index(drop=True).copy()
        # Calculate total raw production length from each row to the end of the order.
        df["produced_length_reverse"]=df["production_m"].fillna(0).iloc[::-1].cumsum().iloc[::-1].values
        # Convert cumulative raw production length into mandrel numbers.
        df["mandrel_no"]=(np.floor(df["produced_length_reverse"]/MANDREL_LENGTH)+1).astype(int)
        # Correct exact boundary cases so that a boundary row belongs to the preceding mandrel.
        boundary_mask=(df["produced_length_reverse"]>0)&((df["produced_length_reverse"]%MANDREL_LENGTH).round(5)==0)
        df.loc[boundary_mask,"mandrel_no"]-=1
        # Mandrel numbering must always start at 1.
        df["mandrel_no"]=df["mandrel_no"].clip(lower=1).astype(int)
        df.drop(columns=["produced_length_reverse"],inplace=True)
        return df
    """
    # ADJUST MANDREL BOUNDARIES USING OD DROPS
    # The theoretical mandrel boundary is refined using actual OD changes in the production data.
    """
    def adjust_mandrel_boundaries(self,df):
        df=df.sort_values("timestamp").reset_index(drop=True).copy()
        # Identify rows where mandrel number changes.
        boundary_mask=df["mandrel_no"].ne(df["mandrel_no"].shift())
        # Store the original theoretical boundaries.
        original_boundaries=df.loc[boundary_mask&(df.index>0),["timestamp","mandrel_no"]].copy()
        if original_boundaries.empty:
            return df,[]
        # Smooth OD to reduce the effect of measurement noise.
        df["OD_smooth"]=pd.to_numeric(df["OD_real"],errors="coerce").rolling(window=5,center=True,min_periods=1).mean()
        # Calculate OD change from one measurement to the next.
        df["od_change"]=df["OD_smooth"].diff()
        all_changes=df["od_change"].dropna().to_numpy()
        """
        # Calculate robust OD-drop threshold.
        # MAD is used instead of standard deviation because it is more resistant to extreme outliers.
        """
        if len(all_changes)==0:
            threshold=MIN_OD_DROP
        else:
            median_change=np.median(all_changes)
            mad=np.median(np.abs(all_changes-median_change))
            robust_sigma=1.4826*mad
            threshold=max(MIN_OD_DROP,DROP_MAD_MULTIPLIER*robust_sigma)
        # Keep only significant negative OD changes.
        all_significant_drops=df[df["od_change"]<=-threshold].copy()
        corrected_boundaries=[]
        # Find the strongest OD drop near every theoretical mandrel boundary.
        for _,boundary in original_boundaries.iterrows():
            original_time=boundary["timestamp"]
            window_start=original_time-pd.Timedelta(minutes=BOUNDARY_SEARCH_MINUTES)
            window_end=original_time+pd.Timedelta(minutes=BOUNDARY_SEARCH_MINUTES)
            nearby_drops=all_significant_drops[(all_significant_drops["timestamp"]>=window_start)&(all_significant_drops["timestamp"]<=window_end)]
            if nearby_drops.empty:
                # No OD drop was found, so retain theoretical boundary.
                corrected_boundaries.append(original_time)
            else:
                # Select the strongest OD decrease.
                strongest_drop=nearby_drops.loc[nearby_drops["od_change"].idxmin()]
                corrected_boundaries.append(strongest_drop["timestamp"])
        """
        # Optimize internal boundaries based on actual production length before and after the boundary.
        """
        optimized_boundaries=corrected_boundaries.copy()
        for i in range(1,len(optimized_boundaries)-1):
            current_boundary=optimized_boundaries[i]
            search_start=current_boundary-pd.Timedelta(minutes=BOUNDARY_SEARCH_MINUTES)
            search_end=current_boundary+pd.Timedelta(minutes=BOUNDARY_SEARCH_MINUTES)
            candidate_drops=all_significant_drops[(all_significant_drops["timestamp"]>=search_start)&(all_significant_drops["timestamp"]<=search_end)]
            if candidate_drops.empty:
                continue
            best_boundary=current_boundary
            best_score=float("inf")
            previous_boundary=optimized_boundaries[i-1]
            next_boundary=optimized_boundaries[i+1]
            # Test every significant OD drop as a possible boundary.
            for _,candidate in candidate_drops.iterrows():
                candidate_time=candidate["timestamp"]
                length_before=self.get_segment_length(df,previous_boundary,candidate_time)
                length_after=self.get_segment_length(df,candidate_time,next_boundary)
                # We want both neighboring mandrels to be close to the nominal mandrel length.
                error_before=abs(length_before-MANDREL_LENGTH)
                error_after=abs(length_after-MANDREL_LENGTH)
                score=error_before+error_after
                if score<best_score:
                    best_score=score
                    best_boundary=candidate_time
            optimized_boundaries[i]=best_boundary
        # Remove duplicate boundaries and sort chronologically.
        corrected_boundaries=sorted(set(optimized_boundaries))
        first_theoretical_boundary=original_boundaries["timestamp"].min()
        first_window_end=first_theoretical_boundary+pd.Timedelta(minutes=BOUNDARY_SEARCH_MINUTES)
        first_section=all_significant_drops[all_significant_drops["timestamp"]<first_window_end]
        if not first_section.empty:
            # Use the earliest significant OD drop in the first boundary region.
            earliest_drop=first_section.sort_values("timestamp").iloc[0]
            first_drop_time=earliest_drop["timestamp"]
            if corrected_boundaries:
                nearest_boundary_distance=min(abs((first_drop_time-b).total_seconds()) for b in corrected_boundaries)
                # If the first drop is sufficiently far away from all existing boundaries, treat it as a boundary.
                if nearest_boundary_distance>BOUNDARY_SEARCH_MINUTES*60:
                    corrected_boundaries.append(first_drop_time)
        corrected_boundaries=sorted(set(corrected_boundaries))
        # Number of mandrels = number of boundaries + 1.
        boundary_count=len(corrected_boundaries)
        total_mandrels=boundary_count+1
        # Initially assign every row to the highest mandrel number.
        df["mandrel_no"] = 1

        for idx, boundary_time in enumerate(corrected_boundaries):
            df.loc[df["timestamp"] >= boundary_time, "mandrel_no"] = idx + 2
        # Ensure valid mandrel numbering.
        df["mandrel_no"]=df["mandrel_no"].clip(lower=1).astype(int)
        # Temporary columns are no longer needed.
        df.drop(columns=["OD_smooth","od_change"],inplace=True,errors="ignore")
        return df,corrected_boundaries
    """
    # REMOVE INITIAL SHORT MANDREL
    # Detects a startup interruption.
    # If production starts, stops shortly afterwards, and the
    # stop lasts long enough, the first/highest mandrel is
    # considered invalid and removed.
    """
    def remove_initial_short_mandrel(self,df):
        if df.empty:
            return df
        # Work on a smaller DataFrame for interruption detection.
        check_df=df[["timestamp","OD_real","mandrel_no"]].copy()
        check_df["timestamp"]=pd.to_datetime(check_df["timestamp"],errors="coerce")
        check_df["OD_real"]=pd.to_numeric(check_df["OD_real"],errors="coerce")
        check_df["mandrel_no"]=pd.to_numeric(check_df["mandrel_no"],errors="coerce")
        check_df=check_df.dropna(subset=["timestamp","OD_real","mandrel_no"]).sort_values("timestamp").reset_index(drop=True)
        if check_df.empty:
            return df
        # Only perform this check when there are enough mandrels.
        current_mandrel_count=int(check_df["mandrel_no"].nunique())
        if current_mandrel_count<=MIN_MANDREL_COUNT_FOR_REMOVAL:
            return df
        # OD > threshold means production is running.
        producing_mask=check_df["OD_real"]>STOP_OD_THRESHOLD
        if not producing_mask.any():
            return df
        # Find the first production point.
        first_production_index=producing_mask.idxmax()
        first_production_time=check_df.loc[first_production_index,"timestamp"]
        # Search after production starts.
        after_production=check_df.loc[check_df.index>first_production_index].copy()
        if after_production.empty:
            return df
        # OD <= threshold means production has stopped.
        interruption_mask=after_production["OD_real"]<=STOP_OD_THRESHOLD
        if not interruption_mask.any():
            return df
        first_interruption_index=interruption_mask.idxmax()
        first_interruption_time=after_production.loc[first_interruption_index,"timestamp"]
        # Duration from production start until interruption.
        interruption_seconds=(first_interruption_time-first_production_time).total_seconds()
        if interruption_seconds<0 or interruption_seconds>INITIAL_INTERRUPTION_MAX_SECONDS:
            return df
        # Find the recovery point after the interruption.
        after_first_interruption=check_df.loc[check_df["timestamp"]>=first_interruption_time].copy()
        recovery_mask=after_first_interruption["OD_real"]>STOP_OD_THRESHOLD
        if recovery_mask.any():
            recovery_index=recovery_mask.idxmax()
            recovery_time=after_first_interruption.loc[recovery_index,"timestamp"]
            stop_duration_seconds=(recovery_time-first_interruption_time).total_seconds()
        else:
            # If production never recovered, measure until the end of the available data.
            stop_duration_seconds=(check_df["timestamp"].iloc[-1]-first_interruption_time).total_seconds()
        # Ignore short interruptions.
        if stop_duration_seconds<MIN_STOP_DURATION_SECONDS:
            return df
        # Highest mandrel is treated as the initial mandrel.
        highest_mandrel=int(check_df["mandrel_no"].max())
        if highest_mandrel<=1:
            return df
        # Remove all rows belonging to the initial short mandrel.
        rows_to_remove=df["mandrel_no"]==highest_mandrel
        removed_count=int(rows_to_remove.sum())
        if removed_count==0:
            return df
        df=df.loc[~rows_to_remove].copy()
        # Renumber remaining mandrels starting from 1 at the end of the order.
        #remaining_highest=int(df["mandrel_no"].max())
        # if remaining_highest>0:
        #     df["mandrel_no"]=remaining_highest-pd.to_numeric(df["mandrel_no"],errors="coerce")+1
        df["mandrel_no"]=pd.to_numeric(df["mandrel_no"],errors="coerce")
        df=df.dropna(subset=["mandrel_no"]).copy()
        df["mandrel_no"]=df["mandrel_no"].clip(lower=1).astype(int)
        print(f"Initial interruption detected after {interruption_seconds:.1f} seconds. Removed highest mandrel {highest_mandrel} ({removed_count:,} rows). Renumbered remaining mandrels.")
        return df
    # REMOVE SHORT LAST MANDREL
    def remove_short_last_mandrel(self,df):
        if df.empty:
            return df
        # Calculate actual production length of every mandrel.
        final_lengths=self.calculate_final_mandrel_lengths(df)
        if final_lengths.empty:
            return df
        # The highest mandrel number is the last mandrel.
        last_mandrel=int(final_lengths["mandrel_no"].max())
        last_length=float(final_lengths.loc[final_lengths["mandrel_no"]==last_mandrel,"final_mandrel_m"].iloc[0])
        # Keep the last mandrel if it is long enough.
        if last_length>=MIN_LAST_MANDREL_LENGTH:
            return df
        # Remove the short last mandrel.
        rows_to_remove=df["mandrel_no"]==last_mandrel
        removed_count=int(rows_to_remove.sum())
        if removed_count==0:
            return df
        df=df.loc[~rows_to_remove].copy()
        remaining_mandrels=sorted(pd.to_numeric(df["mandrel_no"],errors="coerce").dropna().astype(int).unique())
        if not remaining_mandrels:
            return df
        # Rebuild numbering so the final remaining mandrel is 1.
        #mapping={old:new for new,old in enumerate(reversed(remaining_mandrels),start=1)}
        #df["mandrel_no"]=pd.to_numeric(df["mandrel_no"],errors="coerce").map(mapping)
        df=df.dropna(subset=["mandrel_no"]).copy()
        df["mandrel_no"]=df["mandrel_no"].astype(int)
        print(f"Last mandrel {last_mandrel} has calculated length {last_length:.2f} m. Removed it because it is shorter than {MIN_LAST_MANDREL_LENGTH:.2f} m ({removed_count:,} rows). Renumbered remaining mandrels.")
        return df
    """
    # ASSIGN FIXED MANDREL LENGTHS
    # Once mandrel boundaries are finalized, every row belonging
    # to the same mandrel receives the same final mandrel length.
    """
    def assign_fixed_mandrel_lengths(self,df):
        final_lengths=self.calculate_final_mandrel_lengths(df)
        if final_lengths.empty:
            df["mandrel_m"]=np.nan
            return df
        # Create: mandrel number -> final length.
        length_map=final_lengths.set_index("mandrel_no")["final_mandrel_m"].to_dict()
        # Assign the same final length to every row of a mandrel.
        df["mandrel_m"]=df["mandrel_no"].map(length_map)
        # Force the final length into the allowed range.
        df["mandrel_m"]=pd.to_numeric(df["mandrel_m"],errors="coerce").round(2)
        df["mandrel_m"]=df["mandrel_m"].clip(lower=MIN_MANDREL_LENGTH,upper=MAX_MANDREL_LENGTH)
        return df
    # CALCULATE CUMULATIVE ORDER LENGTH
    def calculate_cumulative_order_length(self,df):
        df=df.sort_values("timestamp").reset_index(drop=True).copy()
        # Convert invalid raw production lengths to zero for cumulative calculation.
        df["production_m"]=pd.to_numeric(df["production_m"],errors="coerce").fillna(0)
        # Negative production lengths are not allowed.
        df["production_m"]=df["production_m"].clip(lower=0)
        # Calculate cumulative raw production length of the complete order.
        df["total_m_order"]=df["production_m"].cumsum().round(2)
        return df
    
    # def recalculate_mandrel_numbers_from_forced_lengths(self, df):
    #     mandrel_lengths = (df[["mandrel_no", "mandrel_m"]].drop_duplicates().sort_values("mandrel_no"))
    #     cumulative_boundary = 0.0
    #     for _, row in mandrel_lengths.iterrows():
    #         mandrel_no = int(row["mandrel_no"])
    #         length = float(row["mandrel_m"])
    #         start_boundary = cumulative_boundary
    #         end_boundary = cumulative_boundary + length
    #         df.loc[(df["total_m_order"] > start_boundary) &(df["total_m_order"] <= end_boundary),"mandrel_no"] = mandrel_no
    #         cumulative_boundary = end_boundary
    #     return df^
    def recalculate_mandrel_numbers(self, df):

        mandrels = (
            df[["mandrel_no", "mandrel_m"]]
            .drop_duplicates()
            .sort_values("mandrel_no")
            .reset_index(drop=True)
        )

        boundaries = []
        cumulative = 0.0

        for _, row in mandrels.iterrows():

            cumulative += float(row["mandrel_m"])

            boundaries.append(cumulative)

        df["new_mandrel_no"] = np.searchsorted(
            boundaries,
            df["total_m_order"],
            side="left"
        ) + 1

        df["mandrel_no"] = df["new_mandrel_no"]

        df.drop(columns=["new_mandrel_no"], inplace=True)

        return df
    
    # PROCESS ONE PROGRAM / ORDER
    def process_order(self,program,order):
        # Load raw production data.
        df=self.load_order_data(program,order)
        if df.empty:
            return df
        # Calculate theoretical mandrel numbers.
        df=self.calculate_initial_mandrels(df)
        # Correct mandrel boundaries using OD drops.
        df,_=self.adjust_mandrel_boundaries(df)
        # Remove invalid initial/startup mandrel if required.
        df=self.remove_initial_short_mandrel(df)
        # Remove an incomplete last mandrel if required.
        df=self.remove_short_last_mandrel(df)
        # Assign one fixed length to each mandrel.
        df=self.assign_fixed_mandrel_lengths(df)
        # Calculate cumulative order production length from raw production meters.
        df=self.calculate_cumulative_order_length(df)
        df = self.recalculate_mandrel_numbers(df)
        #df=self.recalculate_mandrel_numbers_from_forced_lengths(df)
        df=self.assign_fixed_mandrel_lengths(df)
        # Rename production speed to the target column name.
        df.rename(columns={"Linie_ist":"line_speed"},inplace=True)
        # Normalize string fields.
        df["compound"]=df["compound"].fillna("").astype(str).str.strip()
        df["Prog_Nr"]=df["Prog_Nr"].fillna("").astype(str).str.strip()
        df["ordername"]=df["ordername"].fillna("").astype(str).str.strip()
        # Columns that must contain numeric values.
        numeric_cols=["ID","wth","OD_real","ID_real","wt_real","line_speed","rpm","mandrel_no","mandrel_m","total_m_order","mass_pressure","mass_temp"]
        # Convert all required numeric columns.
        for col in numeric_cols:
            df[col]=pd.to_numeric(df[col],errors="coerce")
            df[col]=df[col].replace([np.inf,-np.inf],np.nan)
        # Only rows with these mandatory fields are inserted.
        df=df.dropna(subset=["timestamp","OD_real","mandrel_no"]).copy()
        df["mandrel_no"]=df["mandrel_no"].astype(int)
        df["mandrel_m"]=df["mandrel_m"].round(2)
        df["total_m_order"]=df["total_m_order"].round(2)
        # production_m is internal only and is not inserted into the target table.
        df["production_m"]=pd.to_numeric(df["production_m"],errors="coerce").fillna(0).clip(lower=0)
        
        return df
    # FILTER ROWS THAT ARE NOT ALREADY IN TARGET TABLE
    def filter_new_rows(self,df):
        if df.empty:
            return df
        # Build the same unique key used by get_existing_keys().
        keys=list(zip(df["timestamp"],df["Prog_Nr"],df["ordername"]))
        # Keep only rows whose key does not already exist.
        new_mask=np.array([key not in self.existing_keys for key in keys],dtype=bool)
        return df.loc[new_mask].copy()
    # PREPARE DATA FOR INSERT
    def prepare_insert_data(self,df):
        # These columns must match the target table structure.
        columns=["timestamp","Prog_Nr","ordername","compound","ID","wth","OD_real","ID_real","wt_real","line_speed","rpm","mandrel_no","mandrel_m","total_m_order","mass_pressure","mass_temp"]
        insert_df=df[columns].copy()
        row_count=len(insert_df)
        # Assign unique sequential Pos values.
        insert_df.insert(0,"Pos",np.arange(self.next_pos,self.next_pos+row_count,dtype=np.int64))
        # Move the next Pos counter forward.
        self.next_pos+=row_count
        return insert_df
    # INSERT NEW ROWS INTO TARGET TABLE
    def insert_order(self,df):
        if df.empty:
            return 0
        # Add Pos and prepare the final database DataFrame.
        insert_df=self.prepare_insert_data(df)
        # Insert one row only if the same timestamp/program/order combination does not already exist.
        sql="""
INSERT INTO dbo.Ex2_report_dev
([Pos],[timestamp],[Prog_Nr],[ordername],[compound],[ID],[wth],[OD_real],[ID_real],[wt_real],[line_speed],[rpm],[mandrel_no],[mandrel_m],[total_m_order],[mass_pressure],[mass_temp])
SELECT ?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?
WHERE NOT EXISTS
(
SELECT 1
FROM dbo.Ex2_report_dev
WHERE [timestamp]=?
AND CAST([Prog_Nr] AS VARCHAR(255))=?
AND [ordername]=?
)
"""
        total_inserted=0
        cursor=self.connection.cursor()
        #Speed up executemany() for bulk insertion.
        cursor.fast_executemany=True
        # Process data in manageable database batches.
        for start in range(0,len(insert_df),INSERT_BATCH_SIZE):
            batch=insert_df.iloc[start:start+INSERT_BATCH_SIZE]
            rows=[]
            for row in batch.itertuples(index=False,name=None):
                values=list(row)
                # Add the values used by the NOT EXISTS check.
                values.extend([values[1],values[2],values[3]])
                 # Convert pandas NaN values to SQL NULL.
                rows.append(tuple(None if pd.isna(value) else value for value in values))
            # Execute the batch.
            cursor.executemany(sql,rows)
            # Commit after each batch.
            self.connection.commit()
            total_inserted+=len(rows)
            # Add inserted keys to the in-memory set so that
            # subsequent processing does not try to insert them again.
            for row in batch.itertuples(index=False,name=None):
                self.existing_keys.add((row[1],str(row[2]).strip(),str(row[3]).strip()))
        cursor.close()
        return total_inserted
    
    # MAIN PROCESSING LOOP

    def run(self):
         # Load existing target-table keys once at startup.
        self.existing_keys=self.get_existing_keys()
        # Find all program/order combinations that need processing.
        pairs=self.get_prog_order_pairs()
        if pairs.empty:
            print("No Prog_Nr / ordername combinations found.")
            return
        total=len(pairs)
        print(f"Found {total} unique Prog_Nr / ordername combinations.")
        current_program=None
        # Statistics for the final report.
        total_new_rows=0
        new_orders=0
        updated_orders=0
        unchanged_orders=0
        # Process every program/order combination.
        for index,row in pairs.iterrows():
            program=row["Prog_Nr"]
            order=row["ordername"]
            if program!=current_program:
                current_program=program
                print(f"Processing Prog_Nr: {program}")
            print(f"Processing ordername: {order} ({index+1}/{total})")
            try:
                # Check whether this program/order already has any rows in the target table.
                order_already_exists=any(key[1]==program and key[2]==order for key in self.existing_keys)
                 # Run the complete mandrel-processing pipeline.
                df=self.process_order(program,order)
                if df.empty:
                    print("No valid data.")
                    continue
                # Remove rows that have already been inserted.
                new_df=self.filter_new_rows(df)
                if new_df.empty:
                    unchanged_orders+=1
                    print("No new rows. Existing order unchanged.")
                    continue
                 # Insert only new production rows.
                inserted=self.insert_order(new_df)
                total_new_rows+=inserted
                if order_already_exists:
                    updated_orders+=1
                    print(f"New production rows: {inserted:,} | Pos: {self.next_pos-inserted:,}-{self.next_pos-1:,} | Mandrels: {df['mandrel_no'].nunique()}")
                else:
                    new_orders+=1
                    print(f"New order inserted: {inserted:,} rows | Pos: {self.next_pos-inserted:,}-{self.next_pos-1:,} | Mandrels: {df['mandrel_no'].nunique()}")
            except Exception as e:
                # Roll back the current transaction if anything
                # goes wrong for this particular order.
                self.connection.rollback()
                print(f"ERROR: {e}")

        # FINAL SUMMARY

        print("Processing completed.")
        print(f"New rows inserted: {total_new_rows:,}")
        print(f"New orders: {new_orders:,}")
        print(f"Existing orders with new data: {updated_orders:,}")
        print(f"Orders with no new data: {unchanged_orders:,}")
        print(f"Next Pos: {self.next_pos:,}")

# APPLICATION ENTRY POINT

def main():
    conn=None
   
    try:
        # Establish connection to SQL Server.
        conn=pyodbc.connect(CONNECTION_STRING)
        # Create the processing object.
        processor=MandrelProcessor(conn)

        """
         IMPORTANT:
         Uncomment the next line only if you intentionally
         want to delete ALL existing report data first.
        """
        processor.clear_target_table()

        # Start processing
        processor.run()

    except Exception as e:
        # Roll back any open transaction after a fatal error.
        if conn is not None:
            try:
                conn.rollback()
            except Exception:
                pass
        print(f"Fatal error: {e}")
    finally:
        # Always close the database connection.
        if conn is not None:
            conn.close()

# RUN PROGRAM

if __name__=="__main__":
    main()

C:\Users\velagaturi\AppData\Local\Temp\ipykernel_15032\3759907354.py:146: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(EXISTING_ROWS_QUERY,self.connection,params=[START_DATE])
C:\Users\velagaturi\AppData\Local\Temp\ipykernel_15032\3759907354.py:158: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(ORDERS_QUERY,self.connection,params=[START_DATE])


Found 2594 unique Prog_Nr / ordername combinations.
Processing Prog_Nr: 1000
Processing ordername: 6379 (1/2594)


C:\Users\velagaturi\AppData\Local\Temp\ipykernel_15032\3759907354.py:172: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  chunks=pd.read_sql(SOURCE_QUERY,self.connection,params=[START_DATE,program,order],chunksize=CHUNK_SIZE)


No valid data.
Processing Prog_Nr: 1111
Processing ordername: 4008 (2/2594)


C:\Users\velagaturi\AppData\Local\Temp\ipykernel_15032\3759907354.py:172: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  chunks=pd.read_sql(SOURCE_QUERY,self.connection,params=[START_DATE,program,order],chunksize=CHUNK_SIZE)


...

C:\Users\velagaturi\AppData\Local\Temp\ipykernel_15032\3759907354.py:172: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  chunks=pd.read_sql(SOURCE_QUERY,self.connection,params=[START_DATE,program,order],chunksize=CHUNK_SIZE)


No valid data.
Processing ordername: 5104 (45/2594)


C:\Users\velagaturi\AppData\Local\Temp\ipykernel_15032\3759907354.py:172: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  chunks=pd.read_sql(SOURCE_QUERY,self.connection,params=[START_DATE,program,order],chunksize=CHUNK_SIZE)


Initial interruption detected after 142.0 seconds. Removed highest mandrel 113 (88 rows). Renumbered remaining mandrels.
New order inserted: 7,283 rows | Pos: 145,449-152,731 | Mandrels: 113
Processing ordername: 5166 (46/2594)


C:\Users\velagaturi\AppData\Local\Temp\ipykernel_15032\3759907354.py:172: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  chunks=pd.read_sql(SOURCE_QUERY,self.connection,params=[START_DATE,program,order],chunksize=CHUNK_SIZE)
